In [ ]:
import pandas as pd
import pyreadstat
import os
import sys
import numpy as np

Konfigurasi Sistem

In [ ]:

DATA_DIR = 'data'

def load_data(subfolder, filename, required=True):
    path = os.path.join(DATA_DIR, subfolder, filename)
    if not os.path.exists(path):
        msg = f"[SKIP] File tidak ditemukan: {filename}"
        if required:
            print(f"[CRITICAL] {msg}")
        else:
            print(msg)
        return None, None
    
    try:
        df, meta = pyreadstat.read_dta(path)
        df.columns = df.columns.str.lower()
        return df, meta
    except Exception as e:
        print(f"[ERROR] File korup {filename}: {e}")
        return None, None


    

In [ ]:
    # 1. DEMOGRAFI (Usia, Jenis Kelamin)
    
def main():
    print("Mengekstraksi Data IFLS...")
    print("\n[1/7] Memproses Demografi...")
    df_ptrack, _ = load_data('hh14_trk_dta', 'ptrack.dta')
    df_cov, _ = load_data('hh14_b3a_dta', 'b3a_cov.dta')
    
    if df_ptrack is None or df_cov is None: return

    df_demo = df_cov[['pidlink', 'sex', 'age']]
    master_df = pd.merge(df_ptrack[['pidlink']], df_demo, on='pidlink', how='right')
    print(f"   -> Populasi: {len(master_df)} responden.")
    # master_df['age'] = master_df['age'].replace([997, 998, 999], np.nan)    
        # Jalankan ini untuk tes sampling umur di data CD3
    df_test_cd3, _ = load_data('hh14_b3b_dta', 'b3b_cd3.dta')
    if df_test_cd3 is not None:
        # Merge sementara dengan master_df yang sudah punya kolom 'age'
        df_cek = pd.merge(master_df[['pidlink', 'age']], df_test_cd3[['pidlink']].drop_duplicates(), on='pidlink', how='inner')
        print("--- DISTRIBUSI UMUR DI B3B_CD3 ---")
        print(df_cek['age'].describe())
